# Stage C — 25M-class stability and adaptive discovery
Default: an adaptive-only 500-step stability soak. It is exploratory and does not replace the predeclared matched controls required for a causal memory-benefit claim.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='8579b06b641e442191277a61ecfd3644cf1593c4'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
STUDY_PHASE='stability_soak'  # Then use adaptive_discovery only if this phase is finite and resumes correctly.
BLOCK_COUNT=11
D_MODEL=512
NUM_HEADS=8
PERSISTENT_TOKENS=4
MEMORY_DEPTH=1
HORIZON=3
LEARNING_RATE=3e-5
GRADIENT_CLIP_NORM=0.5
VALIDATION_STREAMS=4
PHASES={
    'calibration': {'pilot_name':'c9_25m_model_calibration_50k','valid_base_budget':50_000,'max_optimizer_steps':None,'checkpoint_every':25,'evidence_tier':'engineering','modes':('adaptive','reference','frozen_memory','no_memory'),'run_ids':{mode:'calibration_50k' for mode in ('adaptive','reference','frozen_memory','no_memory')},'amendment':None},
    'stability_soak': {'pilot_name':'c9b_25m_adaptive_stability_500steps','valid_base_budget':None,'max_optimizer_steps':500,'checkpoint_every':100,'evidence_tier':'exploratory','modes':('adaptive',),'run_ids':{'adaptive':'calibration_50k'},'amendment':'adaptive_only_stability_soak_v1'},
    'adaptive_discovery': {'pilot_name':'c10_25m_adaptive_discovery_25mbp','valid_base_budget':25_000_000,'max_optimizer_steps':None,'checkpoint_every':250,'evidence_tier':'exploratory','modes':('adaptive',),'run_ids':{'adaptive':'adaptive_discovery_25m'},'amendment':'adaptive_only_discovery_v1'},
    'primary': {'pilot_name':'c11_25m_model_matched_25mbp','valid_base_budget':25_000_000,'max_optimizer_steps':None,'checkpoint_every':250,'evidence_tier':'confirmatory','modes':('adaptive','reference','frozen_memory','no_memory'),'run_ids':{'adaptive':'adaptive_discovery_25m','reference':'control_reference_equal_budget','frozen_memory':'control_frozen_equal_budget','no_memory':'control_none_equal_budget'},'amendment':None},
}
if STUDY_PHASE not in PHASES: raise ValueError('unknown STUDY_PHASE')
phase=PHASES[STUDY_PHASE]
PILOT_NAME=phase['pilot_name']
VALID_BASE_BUDGET=phase['valid_base_budget']
MAX_OPTIMIZER_STEPS=phase['max_optimizer_steps']
CHECKPOINT_EVERY=phase['checkpoint_every']
EVIDENCE_TIER=phase['evidence_tier']
MODES=phase['modes']
RUN_IDS=phase['run_ids']
AMENDMENT_ID=phase['amendment']

In [ ]:
from pathlib import Path
from google.colab import drive
import json, subprocess,sys
mountpoint=Path('/content/drive')
try:
    if not (mountpoint/'MyDrive').is_dir(): drive.mount(str(mountpoint),force_remount=True,timeout_ms=120000)
except ValueError as error:
    raise RuntimeError('Google Drive did not mount. Restart the Colab runtime, reconnect Drive, accept the authorization prompt, then rerun this cell.') from error
if not (mountpoint/'MyDrive').is_dir(): raise RuntimeError('Google Drive is not ready at /content/drive/MyDrive; do not continue.')
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'remote','set-url','origin',REPO_URL],check=True)
subprocess.run(['git','-C',str(repo),'fetch','--all'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
import torch
device_name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no CUDA device'
if 'A100' not in device_name.upper(): raise RuntimeError(f'Notebook 03 requires an A100; Colab assigned {device_name}. Runtime > Change runtime type, select A100, then restart and rerun.')
selection_path=Path(DRIVE_ROOT)/'runs'/'c1_tokenizers_cpu'/'tokenizer_selection.json'
selection=json.loads(selection_path.read_text(encoding='utf-8'))
selected=selection.get('selected_tokenizer')
if not isinstance(selected,str) or not selected: raise ValueError('tokenizer_selection.json has no selected_tokenizer')
dataset=Path(DRIVE_ROOT)/'stage_c_dataset'/'ordered_streams'/selected
if not (dataset/'token_stream_manifest.json').is_file(): raise FileNotFoundError(f'Run Notebook 00b first; missing {dataset}/token_stream_manifest.json')
DATASET_DIR=str(dataset)
print('Resolved Handoff 00b dataset:',DATASET_DIR)
root=f'{DRIVE_ROOT}/runs/{PILOT_NAME}'
PROTOCOL=repo/'studies'/'stage_c_ecoli_escherichia_medium_25m_v1'/'protocol.json'
STUDY_ROOT=Path(DRIVE_ROOT)/'study'/'stage_c_ecoli_escherichia_medium_25m_v1'
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',root,'--label','hardware_preflight','--repo',str(repo),'--','seqtrainer-titans-stage-c-hardware-preflight','--require','A100'],check=True)

In [ ]:
def show_failure(run_dir, label):
    failure=Path(run_dir)/'FAILED.txt'
    log=Path(run_dir)/'logs'/f'{label}.log'
    if failure.exists(): print(failure.read_text(encoding='utf-8',errors='replace'))
    if log.exists():
        print(f'--- tail of {log} ---')
        print(log.read_text(encoding='utf-8',errors='replace')[-12000:])

if AMENDMENT_ID:
    amendment_path=STUDY_ROOT/'amendments'/f'{AMENDMENT_ID}.json'
    if not amendment_path.exists():
        changes=json.dumps({'phase':STUDY_PHASE,'modes':list(MODES),'valid_base_budget':VALID_BASE_BUDGET,'max_optimizer_steps':MAX_OPTIMIZER_STEPS})
        subprocess.run(['seqtrainer-titans-stage-c-study','amend','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--amendment-id',AMENDMENT_ID,'--rationale','Run adaptive-only exploratory stability/discovery work before matched controls.','--classification','exploratory','--expected-impact','May assess learning and engineering stability; cannot support a causal memory-benefit claim.','--changes',changes],check=True)
stop_args=['--max-valid-bases',str(VALID_BASE_BUDGET)] if VALID_BASE_BUDGET is not None else ['--max-optimizer-steps',str(MAX_OPTIMIZER_STEPS)]
for mode in MODES:
    run_dir=f'{root}/{mode}'
    label=f'train_{mode}'
    command=['seqtrainer-titans-stage-c-train','--dataset-dir',DATASET_DIR,'--run-dir',run_dir,'--memory-mode',mode,'--horizon',str(HORIZON),'--batch-size','1',*stop_args,'--checkpoint-every',str(CHECKPOINT_EVERY),'--learning-rate',str(LEARNING_RATE),'--gradient-clip-norm',str(GRADIENT_CLIP_NORM),'--validation-streams',str(VALIDATION_STREAMS),'--activation','float32','--block-count',str(BLOCK_COUNT),'--d-model',str(D_MODEL),'--num-heads',str(NUM_HEADS),'--persistent-tokens',str(PERSISTENT_TOKENS),'--memory-depth',str(MEMORY_DEPTH),'--protocol',str(PROTOCOL),'--run-id',RUN_IDS[mode]]
    try:
        subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',run_dir,'--label',label,'--repo',str(repo),'--',*command],check=True)
    except subprocess.CalledProcessError:
        show_failure(run_dir,label)
        raise
    subprocess.run(['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--run-id',RUN_IDS[mode],'--evidence-tier',EVIDENCE_TIER,'--artifact',run_dir],check=True)
print('SHARE THIS DIRECTORY:',root)
print('Live status during training: <run>/<mode>/LIVE_STATUS.json')